In [59]:
import pandas as pd
from pathlib import Path
from time import time
import seaborn as sns
import matplotlib.pyplot as plt
import os
import json
import datetime
sys.path.append('./Helpers/')
from utils import import_normalized_dataset, time_discretization,\
                  spatial_filter, spatial_discretization, users_filter,\
                  fill_missing_timestamps, label_home_work_hours, fill_infered_NaN_values,\
                  extract_best_days, relabelize, generate_strings

In [60]:
###### SYSTEM ######
DELETE_USELESS_VARIABLES = True

###### DATASET SELECTION ######
DATASET_NAME = 'yjmob100k' # 'shenzhenurban' or "shanghaikaggle"

###### TEMPORAL DISCRETIZATION ######
TIME_INTERVAL = 30 # minutes

##### SPATIAL FILTERING #####
SPATIAL_FILTERING = False 

##### SPATIAL DISCRETIZATION #####
TILE_DIMENSIONS = (200, 200) # in meters

##### HOME/WORK INFERENCE #####
WORK_BEGIN = 9
WORK_END = 18
WORKPAUSE_BEGIN = 12
WORKPAUSE_END = 14
NIGHT_BEGIN = 23 # Must be BEFORE midnight
NIGHT_END = 7 # Must be AFTER midnight

##### USERS FILTERING (AFTER HOME/WORK COMPLETION) #####
MIN_NUMBER_DAYS = 5
MIN_PERCENTAGE_RECORDS = 0.3
FILTER_NOT_INFERRED_INDIVIDUALS = True

##### FORMATTING #####
NUMBER_OF_BEST_DAYS = 14

##### STAT IMAGE GENERATION #####
# See the cell to set up image parameters

##### FILE SAVING #####
SAVE_COMPLETED_FILTERED_DATASET = True
COMPRESS_SAVEFILE = False

# Dataset Import

In [61]:
# Normalized column names definition
DEVICEID = 'DeviceID'
LATITUDE = 'Latitude'
LONGITUDE = 'Longitude'
TIMESTAMP = 'Timestamp'

LOCATION = 'Location'
WORKTIME = 'Work'
NIGHTTIME = 'Home'

In [62]:
dataset = import_normalized_dataset(dataset_name=DATASET_NAME, convert_timestamps_to_datetime=True, print_duration=True)
# Importing times:
# - geolife: ~4.3s
# - shenzhenurban: ~5.6s
# - yjmob100k: ~22.4s
display(dataset)

Loading dataset from PreprocessedData/0NormalizedCols/yjmob100k.pkl
...
Dataset loaded in 18.37 seconds!

Converting str dates into datetime objects
...
Converted in 298.50 seconds!



,DeviceID,Latitude,Longitude,Timestamp
0,0,79.0,86.0,1900-01-01 00:30:00
1,0,79.0,86.0,1900-01-01 01:00:00
2,0,77.0,86.0,1900-01-01 04:00:00
3,0,77.0,86.0,1900-01-01 04:30:00
4,0,81.0,89.0,1900-01-01 09:30:00
...,...,...,...,...
111535170,99999,119.0,77.0,1900-03-16 19:00:00
111535171,99999,132.0,94.0,1900-03-16 19:30:00
111535172,99999,124.0,105.0,1900-03-16 20:00:00
111535173,99999,121.0,107.0,1900-03-16 20:30:00


In [63]:
# Drop of extra columns ('Altitude'...)
dataset = dataset[[DEVICEID, TIMESTAMP, LATITUDE, LONGITUDE]]

print("Input dataset for the preprocessing pipeline:")
display(dataset)

Input dataset for the preprocessing pipeline:


,DeviceID,Timestamp,Latitude,Longitude
0,0,1900-01-01 00:30:00,79.0,86.0
1,0,1900-01-01 01:00:00,79.0,86.0
2,0,1900-01-01 04:00:00,77.0,86.0
3,0,1900-01-01 04:30:00,77.0,86.0
4,0,1900-01-01 09:30:00,81.0,89.0
...,...,...,...,...
111535170,99999,1900-03-16 19:00:00,119.0,77.0
111535171,99999,1900-03-16 19:30:00,132.0,94.0
111535172,99999,1900-03-16 20:00:00,124.0,105.0
111535173,99999,1900-03-16 20:30:00,121.0,107.0


## Filtering & Discretization

### Spatial Filtering
Spatial filtering of records **only for Geolife** (the only not-already-spatially-discretized dataset) to only Beijing and its borders.

***NOTE**: The spatial filtering is done before the temporal discretization to ensure that points inside the borders are selected.*

In [64]:
print('Original discretized dataset')
nbr_user_before_spatialfilter = dataset[DEVICEID].unique().shape[0]
print(f'Number of user before spatial filtering: {nbr_user_before_spatialfilter}')
display(dataset[[LATITUDE, LONGITUDE]].describe())

nbr_points_before_spatialfilter = dataset.shape[0]

if (DATASET_NAME == 'geolife') and SPATIAL_FILTERING:
    
    spatialfiltered_dataset = spatial_filter(dataset)
    nbr_points_spatialfiltered = nbr_points_before_spatialfilter - spatialfiltered_dataset.shape[0]
    print(f'Number of filtered points: {nbr_points_spatialfiltered} / {nbr_points_before_spatialfilter} points')
    nbr_user_after_spatialfilter = spatialfiltered_dataset[DEVICEID].unique().shape[0]
    print(f'Number of user after spatial filtering: {nbr_user_after_spatialfilter}')
    display(spatialfiltered_dataset[[LATITUDE, LONGITUDE]].describe())
else:
    spatialfiltered_dataset = dataset
if DELETE_USELESS_VARIABLES: del dataset

Original discretized dataset
Number of user before spatial filtering: 100000


,Latitude,Longitude
count,1.115352e+08,1.115352e+08
mean,1.229079e+02,8.565036e+01
std,4.209912e+01,4.281913e+01
min,1.000000e+00,1.000000e+00
25%,9.500000e+01,5.800000e+01
50%,1.270000e+02,8.400000e+01
75%,1.550000e+02,1.110000e+02
max,2.000000e+02,2.000000e+02


### Spatial Discretization

In [65]:
def spatial_discretization(dataset, tile_dimensions, already_discretized=False):
    """Discretize the input dataset into a grid.
    The argument tile_dimensions define the approximate dimensions for each tile.
    
    Args:
        dataset (pandas.DataFrame): Input dataset to discretize.
        tile_dimensions (tuple): Tuple (tile_size_latitude, tile_size_longitude) with the dimensions of a unit of the grid in meters.
        already_discretized (bool, optional): If the dataset is already discretized (ShenzhenUrban & YJMob100K), a particular labelization is performed. Defaults to False.
        
    Returns:
        pandas.DataFrame: Spatially discretized dataset.
        dict: Dictionary of label: coordinates.
        list: [Number of tiles (latitude axis), Number of tiles (longitude axis), Dimension of the latitude axis of a tile (in meters), Dimension of the longitude axis of a tile (in meter)]
    """
    
    LATITUDE_DISTANCE = 778364 # in meters
    LONGITUDE_DISTANCE = (629575 - 569095)/2 + 569095 # in meters
    
    df = dataset.copy()
    
    if already_discretized:
        
        # The ShenzhenUrban and YJMob100K datasets are already discretized
        print("Discretization into bins...")
        labels, bins_locations = pd.factorize(list(zip(df[LATITUDE], df[LONGITUDE])))
        df[LOCATION] = labels + 1 # The value 0 is for NaN
        print("Discretized!")
        
        print("Generation of the label:location dictionary...")
        bins_to_locations = {
            row[LOCATION]: (bins_locations[row[LOCATION] - 1]) for _, row in df.iterrows()
        }
        bins_to_locations = pd.DataFrame.from_dict(bins_to_locations, columns=[LATITUDE, LONGITUDE], orient='index')
        bins_to_locations.rename_axis(LOCATION, inplace=True)
        print("Dictionary generated!")
        
        df.drop(labels=[LATITUDE, LONGITUDE], axis='columns', inplace=True)
        
        return df, bins_to_locations, [None, None, None, None]
    
    # Compute the number of bins necessary for each dimension
    nbr_bins_lat = int(LATITUDE_DISTANCE / tile_dimensions[0])
    nbr_bins_lon = int(LONGITUDE_DISTANCE / tile_dimensions[1])
    
    # Dimensions of a tile
    tile_size_lat = LATITUDE_DISTANCE / nbr_bins_lat
    tile_size_lon = LONGITUDE_DISTANCE / nbr_bins_lon
    
    # Discretize each dimension according to the number_bins value
    print("Discretization into bins...")
    df[LATITUDE], bins_limits_latitude = pd.cut(dataset[LATITUDE], retbins=True,
                                                    labels=list(range(nbr_bins_lat)), bins=nbr_bins_lat)
    df[LATITUDE] = df[LATITUDE].astype('uint32')
    df[LONGITUDE], bins_limits_longitude = pd.cut(dataset[LONGITUDE], retbins=True,
                                                    labels=list(range(nbr_bins_lon)), bins=nbr_bins_lon)
    df[LONGITUDE] = df[LONGITUDE].astype('uint32')
    print("Discretized!")
    
    # Compute the location box number
    print("Computation of the labels...")
    df[LOCATION] = nbr_bins_lon * df[LATITUDE] + df[LONGITUDE] + 1
    print("Labels computed!")
    
    # Create a dictionary label: coordinates
    print("Generation of the label:location dictionary...")
    bins_latitudes = (bins_limits_latitude[df[LATITUDE]] + bins_limits_latitude[df[LATITUDE] + 1]) / 2
    bins_longitudes = (bins_limits_longitude[df[LONGITUDE]] + bins_limits_longitude[df[LONGITUDE] + 1]) / 2
    bins_to_locations = dict(zip(df[LOCATION], zip(bins_latitudes, bins_longitudes)))
    bins_to_locations = pd.DataFrame.from_dict(bins_to_locations, columns=[LATITUDE, LONGITUDE], orient='index')
    bins_to_locations.rename_axis(LOCATION, inplace=True)
    print("Dictionary generated!")
    
    df.drop(labels=[LATITUDE, LONGITUDE], axis='columns', inplace=True)
    
    return df, bins_to_locations, [nbr_bins_lat, nbr_bins_lon, tile_size_lat, tile_size_lon]


In [67]:
dataset_spatial_disc, bins_to_locations, stats_spatial_disc = spatial_discretization(dataset=spatialfiltered_dataset,
                                              tile_dimensions=TILE_DIMENSIONS,
                                              already_discretized=(DATASET_NAME != 'geolife'))

nbr_bins_lat, nbr_bins_lon, tile_size_lat, tile_size_lon = stats_spatial_disc

print("\nSpatially discretized dataset:")
display(dataset_spatial_disc)
print("Dictionary of {label: coordinates}:")
display(bins_to_locations)

if (DATASET_NAME == 'geolife'):
    print(f"Number of tiles in the {nbr_bins_lat} (lat) x {nbr_bins_lon} (lon) grid: {nbr_bins_lat*nbr_bins_lon}")
    print(f"Size of a tile: (latitude, longitude) = ({tile_size_lat:0.2f} m, {tile_size_lon:0.2f} m)")
    
nbr_locations = dataset_spatial_disc[LOCATION].unique().shape[0]

print(f"Total number of unique locations in the set after discretization: {nbr_locations}" +
      (f" / {nbr_bins_lat*nbr_bins_lon}" if (DATASET_NAME == 'geolife') else ""))

print(f"\nFeatures of the dataset after spatial discretization:")
display(pd.DataFrame(dataset_spatial_disc[LOCATION].describe().astype('uint64')))

if DELETE_USELESS_VARIABLES: del spatialfiltered_dataset

Discretization into bins...


/tmp/ipykernel_280364/676742263.py:25: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  labels, bins_locations = pd.factorize(list(zip(df[LATITUDE], df[LONGITUDE])))


Discretized!
Generation of the label:location dictionary...
Dictionary generated!

Spatially discretized dataset:


,DeviceID,Timestamp,Location
0,0,1900-01-01 00:30:00,1
1,0,1900-01-01 01:00:00,1
2,0,1900-01-01 04:00:00,2
3,0,1900-01-01 04:30:00,2
4,0,1900-01-01 09:30:00,3
...,...,...,...
111535170,99999,1900-03-16 19:00:00,4403
111535171,99999,1900-03-16 19:30:00,1474
111535172,99999,1900-03-16 20:00:00,6290
111535173,99999,1900-03-16 20:30:00,15400


Dictionary of {label: coordinates}:


,Latitude,Longitude
Location,,
1,79.0,86.0
2,77.0,86.0
3,81.0,89.0
4,82.0,88.0
5,76.0,86.0
...,...,...
34028,20.0,137.0
34029,136.0,159.0
34030,104.0,184.0


Total number of unique locations in the set after discretization: 34032

Features of the dataset after spatial discretization:


,Location
count,111535175
mean,6714
std,5243
min,1
25%,2334
50%,5627
75%,10025
max,34032


### Temporal Discretization

In [68]:
dataset_temp_disc = time_discretization(dataset_spatial_disc, TIME_INTERVAL)
if DELETE_USELESS_VARIABLES: del dataset_spatial_disc
print("Temporally discretized dataset:")
dataset_temp_disc

Temporally discretized dataset:


,DeviceID,Timestamp,Location
0,0,1900-01-01 00:30:00,1
1,0,1900-01-01 01:00:00,1
2,0,1900-01-01 04:00:00,2
3,0,1900-01-01 04:30:00,2
4,0,1900-01-01 09:30:00,3
...,...,...,...
111535170,99999,1900-03-16 19:00:00,4403
111535171,99999,1900-03-16 19:30:00,1474
111535172,99999,1900-03-16 20:00:00,6290
111535173,99999,1900-03-16 20:30:00,15400


# Home/Work Inference
- Home and work locations inference
	- Identify home and work locations for all the selected users according to the algorithm in the paper and considering the whole life of the users
	- For the 5 selected days what is the ratio of #slots in home location over the #slots in the home period (compute the average and plot the CDF)
	- For all the other days of the user what is the ratio of #slots in home location over the #slots in the home period (compute the average and plot the CDF)
	- For easy comparison plot them on the same graph 
        - The same last 2 plots for work

## Fill missing timestamps with NaN values

Each day with at least one record is filled with NaN records for missing GPS points.

In [69]:
dataset_filled = fill_missing_timestamps(dataset_temp_disc, time_interval=TIME_INTERVAL, chunk_size=250000)
print("Dataset with missing records replaced by 'NaN':")
display(dataset_filled)
# dataset_temp_disc is used later in the notebook as the dataset before completion & filtering
#if DELETE_USELESS_VARIABLES: del dataset_temp_disc

Processing in chunks...


  0%|                                                                              | 0/29 [00:00<?, ?it/s]/home/akouamdj/mobleak-datasets/master_thesis/preprocessing/utils.py:354: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  lambda d: pd.date_range(
  3%|██▍                                                                   | 1/29 [00:57<26:59, 57.85s/it]/home/akouamdj/mobleak-datasets/master_thesis/preprocessing/utils.py:354: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  lambda d: pd.date_range(
  7%|████▊                                                                 | 2/29 [01:58<26:39, 59.26s/it]/home/akouamdj/mobleak-datasets/master_thesis/preprocessing/utils.py:354: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  lambda d: pd.date_range(
 10%|███████▏                                                              | 3/

Concatenating all chunks...
Timestamps filled!
Dataset with missing records replaced by 'NaN':


,DeviceID,Timestamp,Location
0,0,1900-01-01 00:00:00,NaN
1,0,1900-01-01 00:30:00,1.0
2,0,1900-01-01 01:00:00,1.0
3,0,1900-01-01 01:30:00,NaN
4,0,1900-01-01 02:00:00,NaN
...,...,...,...
343085707,99999,1900-03-16 21:30:00,NaN
343085708,99999,1900-03-16 22:00:00,NaN
343085709,99999,1900-03-16 22:30:00,NaN
343085710,99999,1900-03-16 23:00:00,NaN


## Define work/night times

In [70]:
dataset_filled = label_home_work_hours(dataset_filled,
                                       begin_work_hour=WORK_BEGIN, end_work_hour=WORK_END,
                                       begin_workpause_hour=WORKPAUSE_BEGIN, end_workpause_hour=WORKPAUSE_END,
                                       begin_night_hour=NIGHT_BEGIN, end_night_hour=NIGHT_END)
print("Dataset with home/work hours indicated:")
display(dataset_filled)

Dataset with home/work hours indicated:


,DeviceID,Timestamp,Location,Work,Home
0,0,1900-01-01 00:00:00,NaN,False,True
1,0,1900-01-01 00:30:00,1.0,False,True
2,0,1900-01-01 01:00:00,1.0,False,True
3,0,1900-01-01 01:30:00,NaN,False,True
4,0,1900-01-01 02:00:00,NaN,False,True
...,...,...,...,...,...
343085707,99999,1900-03-16 21:30:00,NaN,False,False
343085708,99999,1900-03-16 22:00:00,NaN,False,False
343085709,99999,1900-03-16 22:30:00,NaN,False,False
343085710,99999,1900-03-16 23:00:00,NaN,False,True


### Verification

In [71]:
# Verification: Is there any row with both the Worktime and Nighttime labels
if False:
    print(f"Is there any row with both labels?\n{dataset_filled.apply(lambda row: row[WORKTIME] and row[NIGHTTIME], axis='columns').any()}")

In [72]:
dataset_filled

,DeviceID,Timestamp,Location,Work,Home
0,0,1900-01-01 00:00:00,NaN,False,True
1,0,1900-01-01 00:30:00,1.0,False,True
2,0,1900-01-01 01:00:00,1.0,False,True
3,0,1900-01-01 01:30:00,NaN,False,True
4,0,1900-01-01 02:00:00,NaN,False,True
...,...,...,...,...,...
343085707,99999,1900-03-16 21:30:00,NaN,False,False
343085708,99999,1900-03-16 22:00:00,NaN,False,False
343085709,99999,1900-03-16 22:30:00,NaN,False,False
343085710,99999,1900-03-16 23:00:00,NaN,False,True


## Completion of NaN values

In [73]:
dataset_nanfilled, inferred_locations = fill_infered_NaN_values(dataset_filled)
print("Dataset with 'Nan' values replaced by inferred home/work locations:")
display(dataset_nanfilled)
print("Inferred locations for each individual with at least an inferred location:")
display(inferred_locations)
if DELETE_USELESS_VARIABLES: del dataset_filled

Creating dictionary of most present location at Work for each DEVICEID...
Dictionary created in 37.32s!
Creating the mask for NaN values filling at Work...
Mask created in 0.28s!
Assigning the values through mapping for Work...
Values assigned in 5.73s!

Creating dictionary of most present location at Home for each DEVICEID...
Dictionary created in 34.59s!
Creating the mask for NaN values filling at Home...
Mask created in 0.28s!
Assigning the values through mapping for Home...
Values assigned in 7.35s!

Dataset with 'Nan' values replaced by inferred home/work locations:


,DeviceID,Timestamp,Location
0,0,1900-01-01 00:00:00,2.0
1,0,1900-01-01 00:30:00,1.0
2,0,1900-01-01 01:00:00,1.0
3,0,1900-01-01 01:30:00,2.0
4,0,1900-01-01 02:00:00,2.0
...,...,...,...
343085707,99999,1900-03-16 21:30:00,NaN
343085708,99999,1900-03-16 22:00:00,NaN
343085709,99999,1900-03-16 22:30:00,NaN
343085710,99999,1900-03-16 23:00:00,7371.0


Inferred locations for each individual with at least an inferred location:


,Work,Home
DeviceID,,
0,39.0,2.0
1,135.0,178.0
10,1893.0,1893.0
100,4519.0,4519.0
1000,3900.0,1304.0
...,...,...
99995,861.0,857.0
99996,1723.0,1723.0
99997,186.0,186.0


### Post-completion user filtering

In [74]:
# Filter on users
# Only users with at least n days of at least 30% of records are retained

print(f"Number of users before filtering: {dataset_nanfilled[DEVICEID].unique().shape[0]}")

dataset_filtered, inferred_filtered = users_filter(dataset_nanfilled, TIME_INTERVAL, inferred=inferred_locations,
                                                    min_number_of_days=MIN_NUMBER_DAYS, min_percentage_of_records_per_day=MIN_PERCENTAGE_RECORDS,
                                                    filter_not_inferred=FILTER_NOT_INFERRED_INDIVIDUALS)
print(f"Number of users after post-completion filtering:  {dataset_filtered[DEVICEID].unique().shape[0]}")
display(dataset_filtered)
if DELETE_USELESS_VARIABLES: del dataset_nanfilled

Number of users before filtering: 100000
Number of users after post-completion filtering:  99610


,DeviceID,Timestamp,Location
0,0,1900-01-01 00:00:00,2.0
1,0,1900-01-01 00:30:00,1.0
2,0,1900-01-01 01:00:00,1.0
3,0,1900-01-01 01:30:00,2.0
4,0,1900-01-01 02:00:00,2.0
...,...,...,...
343085707,99999,1900-03-16 21:30:00,NaN
343085708,99999,1900-03-16 22:00:00,NaN
343085709,99999,1900-03-16 22:30:00,NaN
343085710,99999,1900-03-16 23:00:00,7371.0


## Formatting

Conversion of the dataset to the input format of the generative model.

In this format, a line represents an individual like this:

***HOME, WORK, labels of locations for 1 day of an individual***

### Best Days

Extraction of the N best days for each individual.

In [75]:
# N best days extraction
dataset_best_days, dictionary_extracted = extract_best_days(dataset_filtered, bins_to_locations, number_of_best_days=NUMBER_OF_BEST_DAYS)

print(f"Dataset with only the best {NUMBER_OF_BEST_DAYS} day(s):")
display(dataset_best_days)
print(f"Dictionary with only the best {NUMBER_OF_BEST_DAYS} day(s):")
display(dictionary_extracted)

Dataset with only the best 14 day(s):


,DeviceID,Timestamp,Location
0,0,1900-01-04 00:00:00,2.0
1,0,1900-01-04 00:30:00,2.0
2,0,1900-01-04 01:00:00,2.0
3,0,1900-01-04 01:30:00,2.0
4,0,1900-01-04 02:00:00,2.0
...,...,...,...
66937915,99999,1900-03-09 21:30:00,7382.0
66937916,99999,1900-03-09 22:00:00,7371.0
66937917,99999,1900-03-09 22:30:00,NaN
66937918,99999,1900-03-09 23:00:00,7371.0


Dictionary with only the best 14 day(s):


,Latitude,Longitude
Location,,
1,79.0,86.0
2,77.0,86.0
3,81.0,89.0
4,82.0,88.0
5,76.0,86.0
...,...,...
34018,61.0,21.0
34019,59.0,15.0
34020,193.0,132.0


### Relabelization

Convert defined labels from the dataset and the dictionary into lower integers.

In [76]:
import numpy as np
def relabelize(dataset, inferred, dictionary):
    """Relabelization to lower integers.

    Args:
        dataset (pandas.DataFrame): Input dataset.
        inferred (pandas.DataFrame): Input inferred home/work labels dictionary.
        dictionary (pandas.DataFrame): Input mapping of label:coordinates.

    Returns:
        (pandas.DataFrame, pandas.DataFrame, pandas.DataFrame): Returns (dataset, inferred, dictionary) with relabelled location labels.
    """
    
    # Mapper for the relabelization into smaller integers
    relabelization = {old_label: new_label[0] for new_label, old_label in np.ndenumerate(np.concatenate([[np.nan], np.sort(dictionary.index)]))}
    
    # Relabeled inferred home/work labels
    # Infered locations not in the dataset (because remove when selecting N best days are filled with 0)
    inferred_relabeled = inferred.copy()
    inferred_relabeled[WORKTIME] = inferred_relabeled[WORKTIME].map(relabelization).fillna(0).astype('uint64')
    inferred_relabeled[NIGHTTIME] = inferred_relabeled[NIGHTTIME].map(relabelization).fillna(0).astype('uint64')
    
    # Relabeled dictionary
    dictionary_relabeled = dictionary.rename(mapper=relabelization)
    
    # Relabeled dataset
    dataset_relabeled = dataset.copy()
    dataset_relabeled[LOCATION] = dataset_relabeled[LOCATION].map(relabelization)

    return dataset_relabeled, inferred_relabeled, dictionary_relabeled

In [77]:
# Relabelization
dataset_relabeled, inferred_relabeled, dictionary_relabeled = relabelize(dataset_best_days, inferred_filtered, dictionary_extracted)

print(f"Dataset with lower location labels:")
display(dataset_relabeled)
print(f"Inferred locations with lower location labels:")
display(inferred_relabeled)
print(f"Dictionary with lower location labels:")
display(dictionary_relabeled)

Dataset with lower location labels:


,DeviceID,Timestamp,Location
0,0,1900-01-04 00:00:00,2
1,0,1900-01-04 00:30:00,2
2,0,1900-01-04 01:00:00,2
3,0,1900-01-04 01:30:00,2
4,0,1900-01-04 02:00:00,2
...,...,...,...
66937915,99999,1900-03-09 21:30:00,7379
66937916,99999,1900-03-09 22:00:00,7368
66937917,99999,1900-03-09 22:30:00,0
66937918,99999,1900-03-09 23:00:00,7368


Inferred locations with lower location labels:


,Work,Home
DeviceID,,
0,39,2
1,135,178
10,1893,1893
100,4516,4516
1000,3900,1304
...,...,...
99995,861,857
99996,1723,1723
99997,186,186


Dictionary with lower location labels:


,Latitude,Longitude
Location,,
1,79.0,86.0
2,77.0,86.0
3,81.0,89.0
4,82.0,88.0
5,76.0,86.0
...,...,...
30780,61.0,21.0
30781,59.0,15.0
30782,193.0,132.0


### Files generation

Writing of the strings to save in the input files:
- **Trajectories.txt:** HOME, WORK, Trajectory
- **Counts:** HOME WORK: count

In [78]:
# Model's input files content
trajectories_string, prefixes_to_counts = generate_strings(dataset_relabeled, inferred_relabeled)

print("Beginning of trajectories file:")
print(trajectories_string[:1000])
print("\nPrefixes count dictionary:")
print(prefixes_to_counts)

Beginning of trajectories file:
2 39 2 2 2 2 2 2 2 2 2 2 2 2 31 32 33 34 35 36 39 37 38 35 39 40 0 41 39 42 43 44 39 45 46 39 39 39 47 14 8 7 2 0 0 0 0 0 48 2 2 2 2 2 2 2 2 2 2 2 2 2 2 30 33 49 50 51 52 33 53 54 55 39 0 56 57 0 58 57 59 34 34 3 39 31 2 2 0 0 60 61 60 48 2 0 2 2 2 2 2 2 2 2 2 2 2 2 2 2 92 116 117 64 0 118 119 120 107 23 17 121 98 122 64 123 124 125 126 64 75 127 62 128 30 13 129 7 130 13 0 0 0 0 2 89 2 2 2 2 2 2 2 2 2 2 2 31 87 146 147 148 148 149 39 150 151 152 153 151 148 148 150 154 148 149 155 156 148 157 136 92 3 0 89 2 0 0 0 0 0 0 2 2 2 2 2 2 2 2 2 2 2 2 2 2 39 158 107 159 160 145 159 159 161 45 162 39 145 107 163 39 162 39 158 39 55 164 39 45 158 129 7 2 0 0 0 0 0 0 2 2 2 2 2 2 2 2 2 2 2 2 31 180 39 181 120 163 197 39 197 167 172 39 45 107 107 87 39 198 167 33 39 39 39 87 39 61 2 7 2 0 0 0 0 0 0 0 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 62 32 34 39 34 206 207 208 172 176 39 209 210 211 56 39 39 39 142 142 34 39 33 207 212 39 39 92 2 0 0 0 0 2 2 2 2 2 2 2 2 2 2 2 2 2 2 62 2

# Statistics

In [79]:
datasets_paths = {
    'geolife':       'Geolife',
    'shenzhenurban': 'ShenzhenUrban',
    'yjmob100k':     'YJMob100Kv3',
    'shanghaikaggle':     'ShanghaiKaggle',
    }

def define_folder_name(version_number):
    return Path(r"PreprocessedData/1FilteredData") / datasets_paths[DATASET_NAME] / f"{version_number}"

version_number = 0
save_folder = define_folder_name(version_number)
while os.path.exists(save_folder) and len(os.listdir(save_folder)) > 3:
    version_number += 1
    save_folder = define_folder_name(version_number)
    
if not os.path.exists(save_folder): os.makedirs(save_folder)

save_dataset_path = save_folder / 'dataset.pkl'
save_trajectories_string_path = save_folder / 'trajectories.txt'
save_counts_path = save_folder / 'prefixes_to_count.json'
save_inferred_path = save_folder / 'inferred.csv'
save_dictionary_path = save_folder / 'dictionary.csv'
save_description_path = save_folder / 'parameters.txt'
save_statistics_path = save_folder / 'stats.json'
print(f"Version number: {version_number}")

Version number: 0


### Number of full days per individual

In [80]:
# The number of users with an inferred location at the corresponding timespan after filtering
def nbr_inferred_users(dataset, inferred_locs, timespan):
    mask_isfiltered = inferred_locs.index.isin(dataset[DEVICEID].unique())
    mask_isna = ~(inferred_locs[timespan] == 0)

    return (mask_isfiltered & mask_isna).sum()

print("Number of users with an inferred location in the dataset after user filtering & best days selection:")
print(f"- Home: {nbr_inferred_users(dataset_relabeled, inferred_relabeled, NIGHTTIME)} users")
print(f"- Work: {nbr_inferred_users(dataset_relabeled, inferred_relabeled, WORKTIME)} users")

Number of users with an inferred location in the dataset after user filtering & best days selection:
- Home: 99610 users
- Work: 99610 users


In [81]:
# Number of complete days
def nbr_complete_days_per_individual(dataset, time_interval):
    min_number_records = int(1440 / time_interval)
    return (dataset.assign(Date=lambda df: df[TIMESTAMP].dt.date)
    .dropna(axis=0, how='any')
    .groupby(by=[DEVICEID, 'Date'], as_index=False, dropna=True).size()
    .pipe(lambda df: df[df['size']==min_number_records])
    .groupby(by=DEVICEID, as_index=False).size()
    )

complete_days_before = nbr_complete_days_per_individual(dataset_temp_disc, TIME_INTERVAL)
complete_days_after = nbr_complete_days_per_individual(dataset_filtered, TIME_INTERVAL)

print("Number of complete days per individual:")
print("- Before home/work inference, completion of missing values with inferred home/work and filtering:")
display(complete_days_before)
print("- After home/work inference, completion of missing values with inferred home/work and filtering:")
display(complete_days_after)

Number of complete days per individual:
- Before home/work inference, completion of missing values with inferred home/work and filtering:


,DeviceID,size
0,10001,2
1,10215,1
2,10317,1
3,10331,1
4,10370,13
...,...,...
732,99170,17
733,99657,1
734,9967,10
735,99904,1


- After home/work inference, completion of missing values with inferred home/work and filtering:


,DeviceID,size
0,10001,8
1,10013,4
2,10032,1
3,10035,2
4,10037,6
...,...,...
7384,99953,22
7385,99990,5
7386,99991,1
7387,99995,1


In [53]:
# Adds individuals with 0 complete days
def complete_missing_individuals(dataset, all_individuals):
    return pd.merge(dataset, all_individuals, on=DEVICEID, how='right').fillna(0).assign(size=lambda df: df['size'].astype('uint64'))

all_individuals = pd.Series(dataset_temp_disc[DEVICEID].unique(), name=DEVICEID)

complete_days_before = complete_missing_individuals(complete_days_before, all_individuals)
complete_days_after = complete_missing_individuals(complete_days_after, all_individuals)

print("Number of complete days:")
print("- Before completion & filtering")
display(complete_days_before)
print("- After completion & filtering")
display(complete_days_after)

Number of complete days:
- Before completion & filtering


,DeviceID,size
0,0001fa097c95e138d83df164831449d8,0
1,0004dc5b9f3579044a376c2de4513a02,0
2,00070e8f253bed8db1e70c70e6d4fd23,0
3,0010eb6cd724b3d5fda54cbba05c23f6,0
4,001206b37d90ab19ba2eb6bdc3caff05,0
...,...,...
9613,fff50240a8ba03167a15f637c459e27a,0
9614,fff5d915111dac0dc69adde3c0ae8dc4,0
9615,fff8a689be6ad49e0686fe9f7d84a55f,0
9616,fffc459c5a6aefe44560ccaddcab8a00,0


- After completion & filtering


,DeviceID,size
0,0001fa097c95e138d83df164831449d8,0
1,0004dc5b9f3579044a376c2de4513a02,0
2,00070e8f253bed8db1e70c70e6d4fd23,0
3,0010eb6cd724b3d5fda54cbba05c23f6,0
4,001206b37d90ab19ba2eb6bdc3caff05,0
...,...,...
9613,fff50240a8ba03167a15f637c459e27a,0
9614,fff5d915111dac0dc69adde3c0ae8dc4,1
9615,fff8a689be6ad49e0686fe9f7d84a55f,0
9616,fffc459c5a6aefe44560ccaddcab8a00,0


In [82]:
# GRAPH_IMAGE_FOLDER = save_folder
# GRAPH_IMAGE_NAME = f"full_days.png"
# GRAPH_TITLE = f'Full days of traffic'
# GRAPH_XLABEL = "Number of full days"
# GRAPH_YLABEL = "Percentage of individuals"
# COLUMN_TO_PLOT = 'size'

# complete_days = [complete_days_before, complete_days_after]

# sns.set_style('white')
# fig, ax = plt.subplots(figsize=(5, 4))


# plot_colors = ['red', 'blue']
# plot_labels = ['Before completion', 'After completion']

# print(f"Dataset: {DATASET_NAME}")
# for i, df in enumerate(complete_days):
    
#     sns.ecdfplot(data=df, x=COLUMN_TO_PLOT,
#                 lw=3, ax=ax, color=plot_colors[i], label=plot_labels[i])
    
#     #print(f"Number of full days ({time_interval} min): {data_to_plot[COLUMN_TO_PLOT].sum()}")
#     #print(f"Individuals with >= one full day ({time_interval} min): {data_to_plot[data_to_plot[COLUMN_TO_PLOT] > 0].shape[0]}")
    
# ax.set_xscale('log')
# ax.grid(axis='x')
# ax.grid(axis='y')
# ax.set_title(label=GRAPH_TITLE, fontsize=15)
# ax.xaxis.set_tick_params(labelsize=12)
# ax.yaxis.set_tick_params(labelsize=12)
# ax.set_xlabel(GRAPH_XLABEL,  fontsize=15)
# ax.set_ylabel(GRAPH_YLABEL,  fontsize=15)
# plt.legend()
# plt.savefig(GRAPH_IMAGE_FOLDER / GRAPH_IMAGE_NAME, bbox_inches='tight', dpi=300)

# Saving

In [83]:
description_text = \
f"""Version: {version_number}
{datetime.datetime.now().strftime("%d/%m/%Y %H:%M:%S")}

---- Spatial filtering ----
SPATIAL_FILTERING = {SPATIAL_FILTERING}

Number of points filtered: {nbr_points_spatialfiltered if (DATASET_NAME == 'geolife') and SPATIAL_FILTERING else 0} / {nbr_points_before_spatialfilter} points

Number of users:
- before filtering: {nbr_user_before_spatialfilter}
- after filtering:  {nbr_user_after_spatialfilter if (DATASET_NAME == 'geolife') and SPATIAL_FILTERING else ''}

---- Spatial discretization ----
TILE_DIMENSIONS = {"(" + str(TILE_DIMENSIONS[0]) + " m, " + str(TILE_DIMENSIONS[1]) + " m)" if (DATASET_NAME == 'geolife') else ''}

Real tile dimensions (lat, lon): {f"({tile_size_lat:0.2f} m, {tile_size_lon:0.2f} m)" if (DATASET_NAME == 'geolife') else ''}
Number of unique locations in the dataset after discretization: {nbr_locations} / {nbr_bins_lat*nbr_bins_lon if (DATASET_NAME == 'geolife') else nbr_locations} locations

---- Time discretization ----
TIME_INTERVAL = {TIME_INTERVAL} minutes

---- Home/Work inference ----
WORK_BEGIN = {WORK_BEGIN}:00
WORK_END = {WORK_END}:00
WORKPAUSE_BEGIN = {WORKPAUSE_BEGIN}:00
WORKPAUSE_END = {WORKPAUSE_END}:00
NIGHT_BEGIN = {NIGHT_BEGIN}:00
NIGHT_END = {NIGHT_END}:00

---- User filtering AFTER HOME/WORK COMPLETION ----
MIN_NUMBER_DAYS = {MIN_NUMBER_DAYS}
MIN_PERCENTAGE_RECORDS = {MIN_PERCENTAGE_RECORDS}
FILTER_NOT_INFERRED_INDIVIDUALS = {FILTER_NOT_INFERRED_INDIVIDUALS}

Number of users:
- before completion & user filtering: {dataset_temp_disc[DEVICEID].unique().shape[0]}
- after completion & user filtering:  {dataset_filtered[DEVICEID].unique().shape[0]}

Number of users with infered locations after completion & filtering:
- Home inference: {nbr_inferred_users(dataset_filtered, inferred_locations, NIGHTTIME)} / {dataset_filtered[DEVICEID].unique().shape[0]} users
- Work inference: {nbr_inferred_users(dataset_filtered, inferred_locations, WORKTIME)} / {dataset_filtered[DEVICEID].unique().shape[0]} users

Number of user with at least {MIN_NUMBER_DAYS} full days:
- before completion with inferred home/work & filtering: {(complete_days_before['size'] >= MIN_NUMBER_DAYS).sum()}
- after completion & filtering:  {(complete_days_after['size']  >= MIN_NUMBER_DAYS).sum()}

---- Formatting -----
NUMBER_OF_BEST_DAYS = {NUMBER_OF_BEST_DAYS}

Number of unique locations: {dataset_relabeled[LOCATION].unique().shape[0]}

Number of individuals with inferred locations after best days selection:
- Home inference: {(inferred_relabeled[NIGHTTIME] != 0).sum()} / {inferred_relabeled.shape[0]} users
- Work inference: {(inferred_relabeled[WORKTIME] != 0).sum()} / {inferred_relabeled.shape[0]} users

---- File compression ----
Compression method: {('gzip' if COMPRESS_SAVEFILE else 'None')}
"""

with open(save_description_path, 'w') as f:
    f.write(description_text)
print(description_text)

Version: 0
22/07/2025 22:23:23

---- Spatial filtering ----
SPATIAL_FILTERING = False

Number of points filtered: 0 / 111535175 points

Number of users:
- before filtering: 100000
- after filtering:  

---- Spatial discretization ----
TILE_DIMENSIONS = 

Real tile dimensions (lat, lon): 
Number of unique locations in the dataset after discretization: 34032 / 34032 locations

---- Time discretization ----
TIME_INTERVAL = 30 minutes

---- Home/Work inference ----
WORK_BEGIN = 9:00
WORK_END = 18:00
WORKPAUSE_BEGIN = 12:00
WORKPAUSE_END = 14:00
NIGHT_BEGIN = 23:00
NIGHT_END = 7:00

---- User filtering AFTER HOME/WORK COMPLETION ----
MIN_NUMBER_DAYS = 5
MIN_PERCENTAGE_RECORDS = 0.3
FILTER_NOT_INFERRED_INDIVIDUALS = True

Number of users:
- before completion & user filtering: 100000
- after completion & user filtering:  99610

Number of users with infered locations after completion & filtering:
- Home inference: 99610 / 99610 users
- Work inference: 99610 / 99610 users

Number of user with a

In [84]:
# Metrics on the dataset
statistics = {
    'name': DATASET_NAME,
    'version': version_number,
    'time_interval': TIME_INTERVAL,
    'nbr_unique_locations': dataset_relabeled[LOCATION].unique().shape[0],
    'workhours': [i for i in range(WORK_BEGIN, WORK_END) if i not in range(WORKPAUSE_BEGIN, WORKPAUSE_END)],
    'nighthours': [i for i in range(0, NIGHT_END)] + [i for i in range(NIGHT_BEGIN, 24)],
}
pd.DataFrame.from_dict(statistics, orient='index', columns=['Value'])

,Value
name,yjmob100k
version,0
time_interval,30
nbr_unique_locations,30785
workhours,"[9, 10, 11, 14, 15, 16, 17]"
nighthours,"[0, 1, 2, 3, 4, 5, 6, 23]"


In [85]:
if SAVE_COMPLETED_FILTERED_DATASET:
    print("Saving the data...")
    begin = time()
    
    # Dataset (in DataFrame format with timestamps)
    dataset_relabeled.to_pickle(path = save_dataset_path, compression=('gzip' if COMPRESS_SAVEFILE else None))
    
    # Inferred Home/Work
    inferred_relabeled.to_csv(save_inferred_path)
    
    # Dictionary Label:Location
    dictionary_relabeled.to_csv(save_dictionary_path)
    
    # Dataset (in strings format)
    with open(save_trajectories_string_path, 'w') as f:
        f.write(trajectories_string)
    
    # Prefix to counts dictionary
    with open(save_counts_path, 'w') as f:
        json.dump(prefixes_to_counts, f, indent=6)
    
    # Metrics on the dataset
    with open(save_statistics_path, 'w') as f:
        json.dump(statistics, f, indent=6)
        
    end = time()
    print(f"Saved in {end-begin:.2f} seconds.")

Saving the data...
Saved in 8.42 seconds.
